# Phenotype preprocessing

This notebook prepares an analysis-ready phenotype table by:
1. loading the phenotype file
2. defining analysis outcomes
3. applying optional cell-type updates
4. removing rows with missing core cell-type information
5. exporting a cleaned table


In [ ]:
from pathlib import Path

import pandas as pd

PHENO_FILE = Path("data/Pheno_Meth_input.txt")
UPDATED_CELLTYPE_FILE = Path("data/celltype_updates.csv")  # optional
OUTPUT_FILE = Path("data/Pheno_Meth_1_pheno_clean.txt")

CELL_COLS = [
    "EOSINO", "EOSINOpc",
    "LYMPHO", "LYMPHOpc",
    "MONO", "MONOpc",
    "NEUTRO", "NEUTROpc",
    "BASO", "BASOpc",
]

In [ ]:
pheno = pd.read_csv(PHENO_FILE, sep="\t")
print(pheno.shape)
pheno.head()

## Outcome definitions

The notebook creates:
- `AD`
- `AA`
- `AD_only`
- `AA_only`
- `control`
- `AD_and_AA`


In [ ]:
pheno["AD"] = (pheno["PH_DERM"] == 2).astype(int)
pheno["AA"] = (pheno["DX_ASTATO2_2022"] == 2).astype(int)

pheno["AD_only"] = ((pheno["AD"] == 1) & (pheno["AA"] == 0)).astype(int)
pheno["AA_only"] = ((pheno["AA"] == 1) & (pheno["AD"] == 0)).astype(int)
pheno["control"] = ((pheno["AA"] == 0) & (pheno["AD"] == 0)).astype(int)
pheno["AD_and_AA"] = ((pheno["AA"] == 1) & (pheno["AD"] == 1)).astype(int)

print("AD counts:")
print(pheno["AD"].value_counts(dropna=False))
print("\nAA counts:")
print(pheno["AA"].value_counts(dropna=False))

## Optional cell-type updates

If an external update file is available, matching rows are updated by `ID`.


In [ ]:
if UPDATED_CELLTYPE_FILE.exists():
    pheno_update = pd.read_csv(UPDATED_CELLTYPE_FILE)

    pheno_update = pheno_update.dropna(subset=["NEUTROpc"])
    pheno_update = pheno_update[pheno_update["NEUTROpc"] != "na"].copy()

    for col in CELL_COLS:
        if col in pheno_update.columns:
            pheno_update[col] = pd.to_numeric(pheno_update[col], errors="coerce")

    pheno = pheno.set_index("ID")
    pheno_update = pheno_update.set_index("ID")

    common_cols = [col for col in CELL_COLS if col in pheno.columns and col in pheno_update.columns]
    pheno.loc[pheno_update.index, common_cols] = pheno_update[common_cols]
    pheno = pheno.reset_index()

    print("Applied cell-type updates.")
else:
    print(f"Update file not found: {UPDATED_CELLTYPE_FILE}")

## Basic quality checks

In [ ]:
pheno["total_cell_types"] = pheno[
    [col for col in ["EOSINOpc", "LYMPHOpc", "MONOpc", "NEUTROpc", "BASOpc"] if col in pheno.columns]
].sum(axis=1, skipna=False)

print("Missing NEUTROpc:", pheno["NEUTROpc"].isna().sum())
print(pheno["total_cell_types"].describe())

In [ ]:
pheno = pheno.dropna(subset=["NEUTROpc"]).copy()
print("Final shape:", pheno.shape)

In [ ]:
pheno.to_csv(OUTPUT_FILE, sep="\t", index=False)
print(f"Saved cleaned phenotype table to: {OUTPUT_FILE}")